# Quickstart

This notebook walks through the core workflow of SADIE:

1. describe the **water** and the incident **wave**,
2. describe the **vegetation** as a vertical stack of layers,
3. **propagate** the wave across a vegetation patch and inspect the result.

The vegetation here is a simplified rigid-cylinder model (roots and stems) with
dimensions roughly like the 1:10 mangrove model of Zhang et al. (2023). The drag
coefficients are **illustrative values, not fitted ones**; the published Zhang et al.
preset and drag closures arrive in a later release.

Equation numbers refer to the [theory](../theory.md) page.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import sadiepy as sd

print("sadiepy", sd.__version__)

## 1. Water and wave

`Water` gives density and viscosity from temperature (°C) and salinity (g/kg)
(Sharqawy et al. 2010; theory §1). `RegularWave` solves the linear dispersion
relation (theory Eq. 2.2) and provides the derived quantities.

In [ ]:
water = sd.Water(temperature=20.0, salinity=0.0)  # fresh flume water
print(f"rho = {water.rho:.2f} kg/m³, nu = {water.nu:.3e} m²/s")

wave = sd.RegularWave(H=0.08, T=1.4, depth=0.4)
print(f"k = {wave.k:.4f} rad/m, L = {wave.L:.3f} m, kh = {wave.kh:.3f}")
print(f"c = {wave.c:.3f} m/s, c_g = {wave.cg:.3f} m/s")
print(f"Ursell number = {wave.ursell:.2f}, second-order ratio R = {wave.second_order_ratio:.3f}")

The horizontal orbital velocity amplitude $u_m(s)$ (theory Eq. 2.8) grows from the
bed ($s = 0$) to the still-water level ($s = h$).

In [ ]:
s = np.linspace(0.0, wave.depth, 100)
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(wave.u_m(s), s)
ax.axvline(wave.u_bar, ls="--", color="gray", label=r"depth average $\bar u$")
ax.set_xlabel(r"$u_m$ (m/s)")
ax.set_ylabel("height above bed $s$ (m)")
ax.legend()
plt.show()

## 2. Vegetation

Each `Layer` has a vertical extent, a number of elements per m² of bed and a
frontal width per element (a constant, or a function of height). Only the part
of a layer below the still-water level dissipates energy.

In [ ]:
roots = sd.Layer("roots", s_bottom=0.0, s_top=0.10, n_per_m2=1008.0, frontal_width=0.005)
stems = sd.Layer("stems", s_bottom=0.0, s_top=0.40, n_per_m2=36.0, frontal_width=0.010)
veg = sd.Vegetation(layers=(roots, stems), name="rigid roots + stems (illustrative)")

s = np.linspace(0.0, 0.45, 451)
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(veg.frontal_area_profile(s), s)
ax.axhline(wave.depth, color="tab:blue", ls=":", label="still-water level")
ax.set_xlabel("frontal area per unit volume (1/m)")
ax.set_ylabel("height above bed $s$ (m)")
ax.legend()
plt.show()

## 3. Propagate across the patch

With a constant drag coefficient the wave height decays hyperbolically,
$H(x) = H_0 / (1 + \beta x)$ (theory Eq. 3.7), and the damping factors of the layers
add (theory §3.7).

In [ ]:
x = np.linspace(0.0, 2.0, 41)
res = sd.propagate(wave, veg, patch_length=2.0, water=water, drag=1.0, x_out=x)

print(f"method: {res.method}, drag mode: {res.drag_mode}, closures: {res.drag_used}")
print(f"beta = {res.beta:.4f} 1/m, H(2 m)/H0 = {res.H[-1] / wave.H:.3f}")
print("share of dissipation:", {k: round(v, 3) for k, v in res.layer_fractions.items()})

### A wave-dependent drag coefficient

Published closures give $C_D$ as a function of Re, KC or Ur. Here is a **toy
closure for demonstration only** (not a published fit): $C_D = 0.6 + (100/\mathrm{KC})^{1.2}$,
with KC based on the depth-averaged velocity and the stem diameter. Because $C_D$
now changes as the wave decays, the solver integrates the energy balance
numerically (theory Eq. 3.9) and fits an effective $\beta$.

In [ ]:
toy = sd.drag.custom(
    "toy_kc", lambda kc: 0.6 + (100.0 / kc) ** 1.2, variable="KC", length_scale=0.010
)
res_kc = sd.propagate(wave, veg, patch_length=2.0, water=water, drag=toy, x_out=x)

print(
    f"method: {res_kc.method}, effective beta = {res_kc.beta:.4f} 1/m "
    f"(fit RMS {res_kc.beta_fit_rms:.1e})"
)
print("KC range over the patch:", res_kc.dimensionless["KC[toy_kc]"])

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(x, res.H / wave.H, label=r"constant $C_D = 1.0$")
ax.plot(x, res_kc.H / wave.H, label=r"toy $C_D(\mathrm{KC})$")
ax.set_xlabel("distance into patch $x$ (m)")
ax.set_ylabel("$H / H_0$")
ax.legend()
plt.show()

## 4. Validity warnings

SADIE never fails silently outside the range of its theory. A long, high wave in
shallow water violates the second-order nonlinearity criterion (theory Eq. 2.15),
so `RegularWave` emits a `ValidityWarning`, and `propagate` records it in
`result.warnings`.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    steep = sd.RegularWave(H=0.16, T=2.0, depth=0.5)
for w in caught:
    print(w.category.__name__, "-", w.message)